# YOLOv8n training on Colab (T4)
**Runtime → Change runtime type → T4 GPU.** Upload the `yolo-edge-tracker` folder to Google Drive once
(`MyDrive/yolo-edge-tracker`). Code, runs and weights live on **Drive** (they survive disconnects); the
dataset is copied to the **local** `/content` disk (much faster I/O, but it is wiped on disconnect, so re-run cells 3-4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Mount Drive and enter the project
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/yolo-edge-tracker'
%cd $PROJ

In [ ]:
# 2. Install (Colab already has torch + CUDA). TensorBoard logging is off by default in Ultralytics.
!pip install -q ultralytics roboflow albumentations
!yolo settings tensorboard=True

## 3. Get the dataset onto the local disk

In [ ]:
# Option A - straight from Roboflow. Add ROBOFLOW_API_KEY in the Colab Secrets panel (key icon, left sidebar).
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
!python data/download_roboflow.py --workspace YOUR_WORKSPACE --project aerial-objects --version 1 --out /content/raw

# Option B - a zip you exported to Drive:
# !mkdir -p /content/raw && unzip -q -o $PROJ/data/raw/roboflow.zip -d /content/raw

In [ ]:
# 4. Validate + leak-free re-split (one video per split). Writes /content/data/dataset.yaml
!python data/prepare_dataset.py --roboflow /content/raw --out /content/data/dataset --group-pattern "^(.*)_\d+$"
DATA = '/content/data/dataset.yaml'

## 5. Smoke test: 3 epochs on 10% of the data
Checks that everything runs and tells you the seconds per epoch. Multiply by 10 to estimate a full-data epoch.

In [ ]:
!python train/train.py --data $DATA --epochs 3 --fraction 0.1 --project /content/smoke --name smoke

## 6. Baseline training. Start TensorBoard first so you can watch it live

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/train

In [ ]:
!python train/train.py --data $DATA --model yolov8n.pt --epochs 150 --batch 32 --name v8n_640

**Disconnected?** Re-run cells 1-4 (the dataset must be back at the same path), then:

In [ ]:
# !python train/train.py --data $DATA --name v8n_640 --resume

## 7. Visualise curves, confusion matrix and predictions

In [ ]:
from IPython.display import Image, display
!python train/plot_curves.py runs/train/v8n_640
display(Image('runs/train/v8n_640/curves.png'))
display(Image('runs/train/v8n_640/confusion_matrix_normalized.png', width=600))
display(Image('runs/train/v8n_640/val_batch0_pred.jpg', width=900))

## 8. Below 0.65? Improve it, one change at a time
1. Look at the per-class table: if only `drone` is low, the cause is small objects, so try a larger input size:

In [ ]:
!python train/train.py --data $DATA --imgsz 960 --batch 16 --name v8n_960

2. Hyperparameter tuning (long, best run overnight; resumable with `--resume`):

In [ ]:
!python train/tune.py --data $DATA --iterations 30 --epochs 20 --fraction 0.5
!python train/train.py --data $DATA --hyp train/hyp_tuned.yaml --name v8n_640_tuned

In [ ]:
!python train/plot_curves.py runs/train/v8n_640 runs/train/v8n_960 runs/train/v8n_640_tuned --out runs/train/comparison.png
display(Image('runs/train/comparison.png'))

## 9. Export the best model to ONNX for Phase 3

In [ ]:
!pip install -q onnx onnxslim onnxruntime
!python -m export.export_onnx --weights weights/v8n_640.pt --images /content/data/dataset/images/test